# Washington 2008 Presidential Elections: Data Cleaning & Preprocessing

**Goal:** Build a clean, analysis-ready county-level table for Washington, 2008 by merging the presidential primary and presidential general election results, and then derive summary stats (party totals). However, notice that, currently on OpenElections GitHub, there are only data for 3 counties (Adams, Spokane, and Whitman) available for the general election data while there are no information for the primary election.

**Output**: A single CSV where each row is a county and columns include:

- General per-candidate vote counts (prefixed with `gen_`)
- Party totals: `rep_general_total`, `dem_general_total`, `lib_general_total`, `cst_general_total`, `grn_general_total`, `slp_general_total`, `swp_general_total`, `ind_general_total` 

**Last Updated**: 2025/10/11

## 0. Library Import

In [1]:
import re
import pandas as pd
import numpy as np
from pathlib import Path

/Users/amourtu1934/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


## 1. Inputs & Parameters

Define raw file paths once here so the entire notebook is easy to rerun on another machine. If a path changes, we only update it here. We keep a single `OUTPUT_PATH` so all exports land in one known place.

In [3]:
# WA 2008 dataset path
# PRIMARY_PATH = r""
GENERAL_PATH1 = r"../../data/raw/2008/WA/20081104__wa__general__adams__precinct.csv"
GENERAL_PATH2 = r"../../data/raw/2008/WA/20081104__wa__general__spokane__precinct.csv"
GENERAL_PATH3 = r"../../data/raw/2008/WA/20081104__wa__general__whitman__precinct.csv"

# Output directory
OUTPUT_PATH  = r"../../data/processed/2008/WA/"

# Analysis parameters
DISPLAY_ROWS = 10   # Number of rows to display in dataframes

## 2. Load & Filter

We load primary and general datasets separately and immediately subset to the rows we truly need:

- Restrict `office` to 'President' to avoid mixing down-ballot contests

- Remove columns that are fully missing or irrelevant post-filter (e.g., a district column that’s empty for county-level rows)

### b. General Election Dataset

First, we look at the general election data from Adams county.

In [ ]:
# Load general data from Adams
general_df1 = pd.read_csv(GENERAL_PATH1)
general_df1.head(DISPLAY_ROWS)

,county,precinct,office,district,party,candidate,votes
0,Adams,Othello #1,Voters,NaN,NaN,Registered,60
1,Adams,Othello #2,Voters,NaN,NaN,Registered,438
2,Adams,Othello #3,Voters,NaN,NaN,Registered,428
3,Adams,Othello #4,Voters,NaN,NaN,Registered,363
4,Adams,Othello #5,Voters,NaN,NaN,Registered,471
5,Adams,Othello #6,Voters,NaN,NaN,Registered,57
6,Adams,Othello Rural #1,Voters,NaN,NaN,Registered,265
7,Adams,Othello Rural #2,Voters,NaN,NaN,Registered,249
8,Adams,Othello Rural #3,Voters,NaN,NaN,Registered,371
9,Adams,Othello Rural #4,Voters,NaN,NaN,Registered,233


In [5]:
# Different values in 'office' column
general_df1["office"].value_counts()

office
President                       248
State House                     124
Voters                           62
U.S. House                       62
Governor                         62
Lieutenant Governor              62
Secretary of State               62
Treasurer                        62
Auditor                          62
Attorney General                 62
Commissioner of Public Lands     62
Insurance Commissioner           62
State Senate                     31
Name: count, dtype: int64

In [6]:
# Only keep rows where 'office' is 'President'
general_df1 = general_df1[general_df1["office"] == "President"]
general_df1.shape

(248, 7)

In [7]:
# Number of missing values in each column
general_df1.isna().sum()

county         0
precinct       0
office         0
district     248
party          0
candidate      0
votes          0
dtype: int64

We can see that there are no values in the `district` columns. Thus, we can also drop this column besides `office`

In [8]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the district column
general_df1 = general_df1.drop(columns=["office", "district"]).reset_index(drop=True)
general_df1.head(DISPLAY_ROWS)

,county,precinct,party,candidate,votes
0,Adams,RITZ WARD #1,D,Barack Obama,57
1,Adams,RITZ WARD #2,D,Barack Obama,45
2,Adams,RITZ WARD #3,D,Barack Obama,70
3,Adams,RITZ WARD #4,D,Barack Obama,90
4,Adams,RITZ WARD #5,D,Barack Obama,68
5,Adams,RITZ RURAL NW,D,Barack Obama,36
6,Adams,RITZ RURAL SE,D,Barack Obama,32
7,Adams,BATUM,D,Barack Obama,20
8,Adams,BENGE,D,Barack Obama,7
9,Adams,WASHTUCNA #1,D,Barack Obama,30


Now, we aggregate precinct vote counts into county vote counts.

In [9]:
# Make sure votes are numeric
general_df1["votes"] = pd.to_numeric(general_df1["votes"], errors="coerce").fillna(0).astype(int)

# Aggregate precinct vote counts into county vote counts
general_df1 = (
    general_df1.
    groupby(["county", "party", "candidate"], as_index=False)["votes"]
    .sum()
)[["county", "candidate", "party", "votes"]]        # Reorder columns

# Snippet at the aggregated data
general_df1.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Adams,Chuck Baldwin,C,18
1,Adams,Barack Obama,D,1552
2,Adams,Cynthia McKinney,G,2
3,Adams,Ralph Nader,I,44
4,Adams,Bob Barr,L,16
5,Adams,John McCain,R,3222
6,Adams,Gloria La Riva,S&L,2
7,Adams,James E. Harris,SW,2


Now, we move to election data from Spokane county.

In [16]:
# Load general data from Spokane
general_df2 = pd.read_csv(GENERAL_PATH2)
general_df2.head(DISPLAY_ROWS)

,county,precinct,office,district,party,candidate,votes
0,Spokane,3100,REGISTERED VOTERS - TOTAL,NaN,NaN,REGISTERED,731
1,Spokane,3101,REGISTERED VOTERS - TOTAL,NaN,NaN,REGISTERED,625
2,Spokane,3102,REGISTERED VOTERS - TOTAL,NaN,NaN,REGISTERED,911
3,Spokane,3103,REGISTERED VOTERS - TOTAL,NaN,NaN,REGISTERED,636
4,Spokane,3104,REGISTERED VOTERS - TOTAL,NaN,NaN,REGISTERED,781
5,Spokane,3105,REGISTERED VOTERS - TOTAL,NaN,NaN,REGISTERED,731
6,Spokane,3106,REGISTERED VOTERS - TOTAL,NaN,NaN,REGISTERED,788
7,Spokane,3107,REGISTERED VOTERS - TOTAL,NaN,NaN,REGISTERED,950
8,Spokane,3108,REGISTERED VOTERS - TOTAL,NaN,NaN,REGISTERED,1002
9,Spokane,3109,REGISTERED VOTERS - TOTAL,NaN,NaN,REGISTERED,903


In [17]:
# Different values in 'office' column
general_df2["office"].value_counts()

office
President                    2565
State House                  1692
U.S. House                    855
Governor                      855
State Senate                  496
REGISTERED VOTERS - TOTAL     285
BALLOTS CAST - TOTAL          285
Name: count, dtype: int64

In [18]:
# Only keep rows where 'office' is 'President'
general_df2 = general_df2[general_df2["office"] == "President"]
general_df2.shape

(2565, 7)

In [21]:
# Number of missing values in each column
general_df2.isna().sum()

county          0
precinct        0
office          0
district     2565
party           0
candidate       0
votes           0
dtype: int64

We can also drop the `district` column here with the same reason as above.

In [22]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the district column
general_df2 = general_df2.drop(columns=["office", "district"]).reset_index(drop=True)
general_df2.head(DISPLAY_ROWS)

,county,precinct,party,candidate,votes
0,Spokane,3100,D,Barack Obama,268
1,Spokane,3101,D,Barack Obama,249
2,Spokane,3102,D,Barack Obama,412
3,Spokane,3103,D,Barack Obama,279
4,Spokane,3104,D,Barack Obama,343
5,Spokane,3105,D,Barack Obama,342
6,Spokane,3106,D,Barack Obama,355
7,Spokane,3107,D,Barack Obama,390
8,Spokane,3108,D,Barack Obama,424
9,Spokane,3109,D,Barack Obama,372


Now, we aggregate precinct vote counts into county vote counts.

In [23]:
# Make sure votes are numeric
general_df2["votes"] = pd.to_numeric(general_df2["votes"], errors="coerce").fillna(0).astype(int)

# Aggregate precinct vote counts into county vote counts
general_df2 = (
    general_df2.
    groupby(["county", "party", "candidate"], as_index=False)["votes"]
    .sum()
)[["county", "candidate", "party", "votes"]]        # Reorder columns

# Snippet at the aggregated data
general_df2.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Spokane,Chuck Baldwin,C,1499
1,Spokane,Barack Obama,D,105786
2,Spokane,Cynthia McKinney,G,299
3,Spokane,Ralph Nader,I,2635
4,Spokane,Bob Barr,L,870
5,Spokane,Write-ins,NON,1496
6,Spokane,John McCain,R,108314
7,Spokane,Gloria La Riva,S&L,71
8,Spokane,James E. Harris,SW,37


Notice there is a column for write-ins where party is "NON". For simplicity, we can drop this row.

In [46]:
# Drop row where party is NON (write-in candidates)
general_df2 = general_df2.loc[general_df2["party"] != "NON"].copy()
general_df2.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Spokane,Chuck Baldwin,C,1499
1,Spokane,Barack Obama,D,105786
2,Spokane,Cynthia McKinney,G,299
3,Spokane,Ralph Nader,I,2635
4,Spokane,Bob Barr,L,870
6,Spokane,John McCain,R,108314
7,Spokane,Gloria La Riva,S&L,71
8,Spokane,James E. Harris,SW,37


Next, we will clean general election data from Whitman.

In [47]:
# Load general data from Adams
general_df3 = pd.read_csv(GENERAL_PATH3)
general_df3.head(DISPLAY_ROWS)

,county,precinct,office,district,party,candidate,votes
0,Whitman,101 TEKOA,Voters,NaN,NaN,Registration,541
1,Whitman,102 OAKESDALE,Voters,NaN,NaN,Registration,446
2,Whitman,103 FARMINGTON,Voters,NaN,NaN,Registration,150
3,Whitman,104 STEPTOE,Voters,NaN,NaN,Registration,188
4,Whitman,105 GARFIELD,Voters,NaN,NaN,Registration,520
5,Whitman,106 PALOUSE,Voters,NaN,NaN,Registration,296
6,Whitman,107 PALOUSE,Voters,NaN,NaN,Registration,690
7,Whitman,108 COLTON,Voters,NaN,NaN,Registration,474
8,Whitman,109 UNIONTOWN,Voters,NaN,NaN,Registration,282
9,Whitman,110 SOUTH PULLMAN,Voters,NaN,NaN,Registration,702


In [48]:
# Different values in 'office' column
general_df3["office"].value_counts()

office
President                       456
State House                     228
U.S. House                      114
Governor                        114
Lt. Governor                    114
Secretary of State              114
Treasurer                       114
Auditor                         114
Attorney General                114
Commissioner of Public Lands    114
Insurance Commissioner          114
Voters                          113
State Senate                     57
Name: count, dtype: int64

In [49]:
# Only keep rows where 'office' is 'President'
general_df3 = general_df3[general_df3["office"] == "President"]
general_df3.shape

(456, 7)

In [50]:
# Number of missing values in each column
general_df3.isna().sum()

county         0
precinct       0
office         0
district     456
party          0
candidate      0
votes          0
dtype: int64

Now, drop `office` and `district` columns.

In [51]:
# Now, drop the "office" column as it's no longer needed
# Also, drop the district column
general_df3 = general_df3.drop(columns=["office", "district"]).reset_index(drop=True)
general_df3.head(DISPLAY_ROWS)

,county,precinct,party,candidate,votes
0,Whitman,WHITMAN WA,D,Barack Obama,"9,070"
1,Whitman,TEKOA 101,D,Barack Obama,164
2,Whitman,OAKESDALE 102,D,Barack Obama,109
3,Whitman,FARMINGTON 103,D,Barack Obama,58
4,Whitman,STEPTOE 104,D,Barack Obama,50
5,Whitman,GARFIELD 105,D,Barack Obama,180
6,Whitman,PALOUSE 106,D,Barack Obama,105
7,Whitman,PALOUSE 107,D,Barack Obama,328
8,Whitman,COLTON 108,D,Barack Obama,174
9,Whitman,UNIONTOWN 109,D,Barack Obama,97


Now, we aggregate precinct vote counts into county vote counts.

In [52]:
# Make sure votes are numeric
general_df3["votes"] = pd.to_numeric(general_df3["votes"], errors="coerce").fillna(0).astype(int)

# Aggregate precinct vote counts into county vote counts
general_df3 = (
    general_df3.
    groupby(["county", "party", "candidate"], as_index=False)["votes"]
    .sum()
)[["county", "candidate", "party", "votes"]]        # Reorder columns

# Snippet at the aggregated data
general_df3.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Whitman,Chuck Baldwin,C,132
1,Whitman,Barack Obama,D,9070
2,Whitman,Cynthia McKinney,G,52
3,Whitman,Ralph Nader,I,410
4,Whitman,Bob Barr,L,208
5,Whitman,John McCain,R,8104
6,Whitman,Gloria La Riva,S&L,16
7,Whitman,James Harris,SW,12


After cleaning data from each county individually, we combine all these data into one single dataframe `general_df`.

In [53]:
# Combine three separate df into general_df
general_df = pd.concat([general_df1, general_df2, general_df3], ignore_index=True, sort=False)
general_df.head(DISPLAY_ROWS)

,county,candidate,party,votes
0,Adams,Chuck Baldwin,C,18
1,Adams,Barack Obama,D,1552
2,Adams,Cynthia McKinney,G,2
3,Adams,Ralph Nader,I,44
4,Adams,Bob Barr,L,16
5,Adams,John McCain,R,3222
6,Adams,Gloria La Riva,S&L,2
7,Adams,James E. Harris,SW,2
8,Spokane,Chuck Baldwin,C,1499
9,Spokane,Barack Obama,D,105786


In [54]:
# Different values in party column
general_df["party"].value_counts()

party
C      3
D      3
G      3
I      3
L      3
R      3
S&L    3
SW     3
Name: count, dtype: int64

In [55]:
# Shape after preprocessing
general_df.shape

(24, 4)

## 3. Table Pivoting

We convert tall (one row per county/party/candidate) into wide (one row per county with one column per candidate). This creates the consistent schema with previous group cleaned data.

Helper functions:

- `normalize_party(s)`: in this case, we lower everything so column names are stable with other dataframes
- `candidate_token(name)`: turns “Barack Obama” -> OBAMA, “John McCain” -> MCCAIN, etc. Create a short, readable, unique token for column names
- `pivot_wide(df, prefix, key_col="county")`: Main pivot function
        
    * groups by `county` x `party` × `candidate`, sums `votes`,
    * pivots to columns named like:
        * Primary: `pri_dem_OBAMA`, `pri_rep_MCCAIN`,...
        * General: `gen_dem_OBAMA`, `gen_rep_MCCAIN`,...

    * flattens the MultiIndex into plain column strings,
    * returns one wide row per county

In [56]:
def normalize_party(s: pd.Series) -> pd.Series:
    """
    Normalize party names: Democratic -> dem, Republican -> rep
    """
    return(s.str.strip()
           .str.capitalize()
           .map({
                "D"     : "dem", 
                "R"     : "rep",
                "L"     : "lib",
                "C"     : "cst",
                "G"     : "grn",
                "S&l"   : "slp",
                "Sw"    : "swp",
                "I"     : "ind",
               })
           .fillna(s.str.strip().str.lower()))      # For defensive purposes only, would not expect other parties

In [57]:
SUFFIXES = {
    "JR","SR","JNR","SNR",
    "II","III","IV","V","VI","VII","VIII","IX","X","XI","XII"
}

def candidate_token(name: str) -> str:
    """
    Turn John McCain -> MCCAIN, Barack Obama -> OBAMA
    Skip suffixes, keep last name/token, capitalize, and remove punctuation
    """
    if pd.isna(name):
        return "UNKNOWN"                # Defensive purposes only, would not expect missing values
    
    # Remove suffixes
    raw = str(name).strip()

    # If a comma exists, treat as 'LAST, FIRST ...'
    if "," in raw:
        last_part = raw.split(",", 1)[0]
        last_part = re.sub(r"[^A-Za-z0-9\s]+", "", last_part).strip().upper()
        tokens = last_part.split()
        return tokens[-1] if tokens else "UNKNOWN"

    # Otherwise: remove punctuation, split, then drop trailing suffixes
    tokens = re.sub(r"[^A-Za-z0-9\s]+", "", raw).strip().upper().split()
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()
    return tokens[-1] if tokens else "UNKNOWN"

In [58]:
def pivot_wide(df: pd.DataFrame, prefix: str, key_col: str="county") -> pd.DataFrame:
    """
    Pivot the dataframe to wide format based on party and candidate
    """
    # Normalize party names
    df['party_key'] = normalize_party(df['party'])
    
    # Create candidate tokens
    df['candidate_token'] = df['candidate'].apply(candidate_token)
    
    # Create new column names based on party and candidate token
    df['new_col'] = prefix + '_' + df['party'] + '_' + df['candidate_token']
    
    # Pivot the dataframe
    pivot_df = df.pivot_table(index=key_col, 
                              columns=["party_key", "candidate_token"], 
                              values="votes", 
                              aggfunc='sum', 
                              fill_value=0)
    
    # Flatten multi-level columns
    pivot_df.columns = [f"{prefix}_{p}_{c}" for p, c in pivot_df.columns]
    
    # Reset index to turn key_col back into a column
    pivot_df = pivot_df.reset_index()
    
    return pivot_df

In [59]:
# General dataframe pivot
general_pivot = pivot_wide(general_df, prefix="gen")
general_pivot.head(DISPLAY_ROWS)

,county,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN,gen_slp_RIVA,gen_swp_HARRIS
0,Adams,18,1552,2,44,16,3222,2,2
1,Spokane,1499,105786,299,2635,870,108314,71,37
2,Whitman,132,9070,52,410,208,8104,16,12


## 4. Adding Party Total Columns

Now, we will add party totals columns for general totals:

* `rep_general_total` = sum of all `gen_rep_*` columns
* `dem_general_total` = sum of all `gen_dem_*` columns
* `lib_general_total` = sum of all `gen_lib_*` columns
* `cst_general_total` = sum of all `gen_cst_*` columns
* `grn_general_total` = sum of all `gen_grn_*` columns
* `slp_general_total` = sum of all `gen_slp_*` columns
* `swp_general_total` = sum of all `gen_swp_*` columns
* `ind_general_total` = sum of all `gen_ind_*` columns

In [60]:
# Add party totals for general election
rep_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_rep")] 
dem_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_dem")]
lib_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_lib")]
cst_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_cst")]
grn_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_grn")]
slp_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_slp")]
swp_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_swp")]
ind_general_cols    = [c for c in general_pivot.columns if c.startswith("gen_ind")]


general_pivot["rep_general_total"] = general_pivot[rep_general_cols].sum(axis=1) if rep_general_cols else 0
general_pivot["dem_general_total"] = general_pivot[dem_general_cols].sum(axis=1) if dem_general_cols else 0
general_pivot["lib_general_total"] = general_pivot[lib_general_cols].sum(axis=1) if lib_general_cols else 0
general_pivot["cst_general_total"] = general_pivot[cst_general_cols].sum(axis=1) if cst_general_cols else 0
general_pivot["grn_general_total"] = general_pivot[grn_general_cols].sum(axis=1) if grn_general_cols else 0
general_pivot["slp_general_total"] = general_pivot[slp_general_cols].sum(axis=1) if slp_general_cols else 0
general_pivot["swp_general_total"] = general_pivot[swp_general_cols].sum(axis=1) if swp_general_cols else 0
general_pivot["ind_general_total"] = general_pivot[ind_general_cols].sum(axis=1) if ind_general_cols else 0

In [61]:
# Print out all the column names in the final dataframe
print("Final columns in the cleaned general dataframe:")
general_pivot.columns

Final columns in the cleaned general dataframe:


Index(['county', 'gen_cst_BALDWIN', 'gen_dem_OBAMA', 'gen_grn_MCKINNEY',
       'gen_ind_NADER', 'gen_lib_BARR', 'gen_rep_MCCAIN', 'gen_slp_RIVA',
       'gen_swp_HARRIS', 'rep_general_total', 'dem_general_total',
       'lib_general_total', 'cst_general_total', 'grn_general_total',
       'slp_general_total', 'swp_general_total', 'ind_general_total'],
      dtype='object')

In [62]:
# Preview the general_pivot dataframe with totals
general_pivot.head(DISPLAY_ROWS)

,county,gen_cst_BALDWIN,gen_dem_OBAMA,gen_grn_MCKINNEY,gen_ind_NADER,gen_lib_BARR,gen_rep_MCCAIN,gen_slp_RIVA,gen_swp_HARRIS,rep_general_total,dem_general_total,lib_general_total,cst_general_total,grn_general_total,slp_general_total,swp_general_total,ind_general_total
0,Adams,18,1552,2,44,16,3222,2,2,3222,1552,16,18,2,2,2,44
1,Spokane,1499,105786,299,2635,870,108314,71,37,108314,105786,870,1499,299,71,37,2635
2,Whitman,132,9070,52,410,208,8104,16,12,8104,9070,208,132,52,16,12,410


Now, we save the cleaned dataframe into the processed directory.

In [63]:
# Save the cleaned and merged dataframe to CSV
out_dir = Path(OUTPUT_PATH)
out_dir.mkdir(parents=True, exist_ok=True)
general_pivot.to_csv(OUTPUT_PATH + "WA.csv", index=False)